In [ ]:
# Kaggle GRU training (seq2seq) — GPU + BEST checkpoint saving + early stopping + grad clipping
# Paste this as ONE cell and run.

import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ======================
# CONFIG
# ======================
TRAIN_FILE = Path("/kaggle/input/lob-data/train.parquet")
VALID_FILE = Path("/kaggle/input/lob-data/valid.parquet")

OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Training
BATCH_SIZE = 32
EPOCHS = 20
LR = 1e-3
NUM_WORKERS = 2      # try 0 / 2 / 4
PATIENCE = 3         # early stopping
CLIP_NORM = 1.0      # gradient clipping

# Model
INPUT_DIM = 32
HIDDEN = 128
NUM_LAYERS = 6
DROPOUT = 0.1
D_OUT = 2

SEED = 42


def seed_all(seed: int = 42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class SeqDataset(Dataset):
    """
    Returns full sequences per seq_ix:
      X: [T, 32], Y: [T, 2], N: [T]
    """
    def __init__(self, df: pd.DataFrame):
        self.groups = []
        for _, g in df.groupby("seq_ix", sort=False):
            g = g.sort_values("step_in_seq")

            # NOTE: assumes fixed column order (as in competition files)
            x = g.iloc[:, 3:35].to_numpy(dtype=np.float32)     # 32 feats
            y = g.iloc[:, 35:37].to_numpy(dtype=np.float32)    # 2 targets
            n = g["need_prediction"].to_numpy(dtype=np.uint8)  # [T]
            self.groups.append((x, y, n))

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, idx):
        x, y, n = self.groups[idx]
        return torch.from_numpy(x), torch.from_numpy(y), torch.from_numpy(n)


def collate_stack(batch):
    xs, ys, ns = zip(*batch)
    X = torch.stack(xs, dim=0)  # [B, T, 32]
    Y = torch.stack(ys, dim=0)  # [B, T, 2]
    N = torch.stack(ns, dim=0)  # [B, T]
    return X, Y, N


class GRUModel(nn.Module):
    def __init__(self, input_dim=32, hidden=128, num_layers=6, dropout=0.1, d_out=2):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden,
            num_layers=num_layers,
            dropout=dropout if num_layers >= 2 else 0.0,
            batch_first=True,
        )
        self.head = nn.Linear(hidden, d_out)

    def forward(self, x):
        h, _ = self.gru(x)      # [B, T, H]
        return self.head(h)     # [B, T, 2]


def weighted_mse(pred, target):
    w = target.abs().clamp_min(1e-3)
    return ((pred - target) ** 2 * w).mean()


def main():
    seed_all(SEED)

    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu:", torch.cuda.get_device_name(0))
    print("DEVICE:", DEVICE)

    print("Loading train + valid...")
    train_df = pd.read_parquet(TRAIN_FILE)
    valid_df = pd.read_parquet(VALID_FILE)

    train_ds = SeqDataset(train_df)
    valid_ds = SeqDataset(valid_df)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == "cuda"),
        collate_fn=collate_stack,
    )
    valid_loader = DataLoader(
        valid_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == "cuda"),
        collate_fn=collate_stack,
    )

    model = GRUModel(
        input_dim=INPUT_DIM,
        hidden=HIDDEN,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
        d_out=D_OUT,
    ).to(DEVICE)

    opt = torch.optim.Adam(model.parameters(), lr=LR)

    print(
        f"BATCH_SIZE={BATCH_SIZE} | EPOCHS={EPOCHS} | LR={LR} | "
        f"HIDDEN={HIDDEN} | LAYERS={NUM_LAYERS} | DROPOUT={DROPOUT} | "
        f"PATIENCE={PATIENCE} | CLIP_NORM={CLIP_NORM}"
    )

    total_start = time.time()

    best_valid = float("inf")
    bad_epochs = 0

    best_path = OUT_DIR / f"gru_best_h{HIDDEN}_L{NUM_LAYERS}_do{DROPOUT}.pt"
    last_path = OUT_DIR / f"gru_last_h{HIDDEN}_L{NUM_LAYERS}_do{DROPOUT}.pt"

    valid_avg = None  # filled after first epoch

    for epoch in range(EPOCHS):
        epoch_start = time.time()

        # ---- train ----
        model.train()
        train_loss_sum = 0.0

        for X, Y, N in train_loader:
            X = X.to(DEVICE, non_blocking=True)
            Y = Y.to(DEVICE, non_blocking=True)
            N = N.to(DEVICE, non_blocking=True)

            pred = model(X)  # [B, T, 2]
            mask = (N == 1)

            loss = weighted_mse(pred[mask], Y[mask])

            opt.zero_grad(set_to_none=True)
            loss.backward()

            if CLIP_NORM is not None and CLIP_NORM > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)

            opt.step()
            train_loss_sum += loss.item()

        # ---- valid ----
        model.eval()
        with torch.no_grad():
            valid_loss_sum = 0.0
            for X, Y, N in valid_loader:
                X = X.to(DEVICE, non_blocking=True)
                Y = Y.to(DEVICE, non_blocking=True)
                N = N.to(DEVICE, non_blocking=True)

                pred = model(X)
                mask = (N == 1)
                valid_loss_sum += weighted_mse(pred[mask], Y[mask]).item()

        train_avg = train_loss_sum / len(train_loader)
        valid_avg = valid_loss_sum / len(valid_loader)

        # ---- save BEST + early stopping ----
        saved_best = ""
        if valid_avg < best_valid:
            best_valid = valid_avg
            bad_epochs = 0

            torch.save(
                {
                    "state_dict": model.state_dict(),
                    "input_dim": INPUT_DIM,
                    "hidden": HIDDEN,
                    "num_layers": NUM_LAYERS,
                    "dropout": DROPOUT,
                    "epoch": epoch + 1,
                    "best_valid_loss": best_valid,
                },
                best_path,
            )
            saved_best = "  -> saved BEST"
        else:
            bad_epochs += 1

        epoch_time = time.time() - epoch_start
        eta = epoch_time * (EPOCHS - (epoch + 1))

        print(
            f"epoch {epoch+1}/{EPOCHS}: "
            f"train_loss={train_avg:.4f} "
            f"valid_loss={valid_avg:.4f} "
            f"| time={epoch_time:.1f}s ETA={eta/60:.1f}m"
            f"{saved_best}"
        )

        if bad_epochs >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1} (no improvement for {PATIENCE} epochs).")
            break

    # ---- save LAST ----
    torch.save(
        {
            "state_dict": model.state_dict(),
            "input_dim": INPUT_DIM,
            "hidden": HIDDEN,
            "num_layers": NUM_LAYERS,
            "dropout": DROPOUT,
            "epoch": (epoch + 1),
            "valid_loss_last": float(valid_avg) if valid_avg is not None else None,
        },
        last_path,
    )

    print("Saved BEST:", best_path)
    print("Saved LAST:", last_path)
    print(f"Total train time: {(time.time() - total_start)/60:.1f} minutes")


if __name__ == "__main__":
    main()

Check path for utils.py

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn

# Add the Kaggle "Model" input folder that contains utils.py
sys.path.append("/kaggle/input/gru2/other/default/1")

from utils import DataPoint, ScorerStepByStep

DEVICE = "cpu"  # submission is CPU, scoring on CPU is fine

class GRUModel(nn.Module):
    def __init__(self, input_dim=32, hidden=128, num_layers=4, dropout=0.1, d_out=2):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden,
            num_layers=num_layers,
            dropout=dropout if num_layers >= 2 else 0.0,
            batch_first=True,
        )
        self.head = nn.Linear(hidden, d_out)

    def forward(self, x, h=None):
        out, h = self.gru(x, h)
        y = self.head(out)
        return y, h

class PredictionModel:
    def __init__(self, ckpt_path: str):
        ckpt = torch.load(ckpt_path, map_location=DEVICE)

        self.model = GRUModel(
            input_dim=int(ckpt.get("input_dim", 32)),
            hidden=int(ckpt.get("hidden", 128)),
            num_layers=int(ckpt.get("num_layers", 4)),
            dropout=float(ckpt.get("dropout", 0.1)),
            d_out=2,
        ).to(DEVICE)

        self.model.load_state_dict(ckpt["state_dict"])
        self.model.eval()

        self.current_seq_ix = None
        self.h = None

    def _reset(self, seq_ix: int):
        self.current_seq_ix = seq_ix
        self.h = None

    @torch.no_grad()
    def predict(self, data_point: DataPoint):
        if self.current_seq_ix != data_point.seq_ix:
            self._reset(data_point.seq_ix)

        x = torch.from_numpy(data_point.state.astype(np.float32, copy=False)).to(DEVICE).view(1, 1, -1)
        y, self.h = self.model(x, self.h)
        pred = y[0, 0].cpu().numpy().astype(np.float32)

        if not data_point.need_prediction:
            return None

        return np.clip(pred, -6.0, 6.0).astype(np.float32)

# pick BEST checkpoint automatically
best = sorted(Path("/kaggle/working").glob("gru_best_*.pt"))
last = sorted(Path("/kaggle/working").glob("gru_last_*.pt"))

ckpt_path = str(best[-1] if best else last[-1])
print("Using ckpt:", ckpt_path)

test_file = "/kaggle/input/lob-data/valid.parquet"

model = PredictionModel(ckpt_path)
scorer = ScorerStepByStep(test_file)

print("Scoring...")
results = scorer.score(model)
print("Results:", results)